# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs and socio-demographic data related to knowledge adoption and rangeland management in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all available record sets and their fields using their `@id`s.

In [ ]:
record_sets = list(dataset.record_sets)
if not record_sets:
    print('No record sets declared explicitly in the schema metadata. Fetching from dataset contents...')
    # List possible record sets by iterating dataset.record_sets generator
    record_set_ids = set()
    for rs in dataset.record_sets:
        print(f"Record set: @id='{rs.id}', name='{rs.name}'")
        record_set_ids.add(rs.id)
    record_sets = list(record_set_ids)
else:
    print('Available record sets:')
    for rs in record_sets:
        print(f"@id: {rs.id} | name: {rs.name}")

### Fields Overview for Each Record Set
For each available record set, we enumerate its fields and columns.

In [ ]:
# List out the record set IDs and their fields (with their @id)
record_set_id_list = []
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs.name} (id: {rs.id})")
    record_set_id_list.append(rs.id)
    try:
        for field in rs.fields:
            print(f"  Field: {field.name} (id: {field.id}) - dataType: {field.data_type}")
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    print(f"    Column: {col.name} (id: {col.id}), type: {col.data_type}")
    except Exception as e:
        print("  (Could not access fields for this record set.)", e)

# Show all discovered record set IDs
print("\nRecord sets discovered (by @id):", record_set_id_list)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing each by its `@id`.

In [ ]:
dataframes = {}
record_set_ids = record_set_id_list
if not record_set_ids:
    print("No record set IDs found: cannot extract data.")
else:
    for record_set_id in record_set_ids:
        print(f"Extracting records for record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id}. Shape: {df.shape}")
            print("Columns:", df.columns.tolist())
        else:
            print(f"No records found for record set @id: {record_set_id}")
# Display a preview of the first available DataFrame
for k, v in dataframes.items():
    print(f"\nFirst 5 rows for record set {k}:")
    display(v.head(5))
    break  # show only the first as preview

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on a numeric field, normalizing, and grouping. All operations use the `@id` for fields and record sets.

In [ ]:
# Provide automatic field selection if possible
main_rs_id = None
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    df_main = dataframes[main_rs_id]
    print(f"Using record set: {main_rs_id}")
    print(f"Available columns: {df_main.columns.tolist()}")
else:
    print('No dataframes loaded for EDA.')

# Attempt to select a numeric field by detecting numeric columns
numeric_field_id = None
group_field_id = None
if main_rs_id is not None:
    numerics = df_main.select_dtypes(include='number').columns.tolist()
    if numerics:
        numeric_field_id = numerics[0]
        print(f"Automatically selected numeric field (by column name/@id): {numeric_field_id}")
    else:
        print("No numeric fields found, EDA will not run.")
    # Try to group by first non-numeric
    for c in df_main.columns:
        if c != numeric_field_id and df_main[c].dtype == object:
            group_field_id = c
            print(f"Automatically selected group field: {group_field_id}")
            break

# EDA example: filtering, normalization, grouping
if main_rs_id is not None and numeric_field_id is not None:
    threshold = df_main[numeric_field_id].quantile(0.75)  # Use 75th percentile for the example
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping if possible
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('Cannot perform EDA: Numeric field not found.')

## 5. Visualization
Visualize data distributions or relationships. Here we plot the distribution of the selected numeric field and a grouped summary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_rs_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,4))
        # Barplot of averages by group
        if 'grouped_df' in locals():
            sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
            plt.title(f'Mean {numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
In this notebook, you have learned how to load, explore, filter, and visualize data from a Croissant-compliant dataset (FAIR^2) using `mlcroissant`.

**Key findings:**
- The dataset contains record sets with regression outputs and household survey fields for rangeland management in Northern Kenya.
- We demonstrated extraction and filtering by field `@id`, normalized numeric variables, and aggregated results by group.
- Visualization provides quick insight into the distribution and key groupwise statistics.

Next, you can further refine your analysis or adapt this notebook to other Croissant datasets.